# 📖 Novel-TUI — Servidor LLM Remoto (GPU T4 en Google Colab con Ngrok)

Este cuaderno ejecuta **KoboldCpp** con aceleración CUDA en GPU T4, el modelo **NeuralDaredevil 8B Abliterated** (0% censura para literatura y R-18) y un **Túnel Fijo Permanente de Ngrok**.

### ⚡ Ventajas:
1. **Dominio Fijo Permanente:** No cambia nunca (`unboasted-nonshattering-mercedes.ngrok-free.dev`).
2. **0% Censura:** Basado en NeuralDaredevil Abliterated sin vector de rechazo.
3. **Persistencia en Google Drive:** El modelo se descarga una sola vez en tu Drive (`/NovelTUI_Models/`) y en los próximos arranques inicia en 5 segundos.

In [ ]:
#@title 🚀 Iniciar Servidor KoboldCpp con GPU, Google Drive y Túnel Fijo Ngrok
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

# 1. Configuración de Ngrok y Rutas
NGROK_TOKEN = '36ZRl5RYYYgkTeOtpt7x6V2hGGe_24UoEWyTvpQmDxQ5aDfT8'
NGROK_DOMAIN = 'unboasted-nonshattering-mercedes.ngrok-free.dev'

DRIVE_DIR = '/content/drive/MyDrive/NovelTUI_Models'
MODEL_PATH = os.path.join(DRIVE_DIR, 'NeuralDaredevil-8B-abliterated-Q5_K_M.gguf')
MODEL_URL = 'https://huggingface.co/bartowski/NeuralDaredevil-8B-abliterated-GGUF/resolve/main/NeuralDaredevil-8B-abliterated-Q5_K_M.gguf'
KOBOLD_URL = 'https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64'

!mkdir -p /content/novel-llm
!mkdir -p "{DRIVE_DIR}"
%cd /content/novel-llm

# 2. Instalar Ngrok oficial y configurar token
if not os.path.exists('/usr/local/bin/ngrok'):
    print('📥 Instalando Ngrok...')
    !wget -q -c https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -O ngrok.tgz
    !tar -xzf ngrok.tgz -C /usr/local/bin
    !rm -f ngrok.tgz

!ngrok config add-authtoken {NGROK_TOKEN}

# 3. Iniciar Túnel Ngrok con tu dominio estático en segundo plano
print(f'⚡ Iniciando túnel estático: https://{NGROK_DOMAIN}')
!killall ngrok 2>/dev/null || true
subprocess.Popen(['ngrok', 'http', '5001', f'--url={NGROK_DOMAIN}']) 

# 4. Descargar KoboldCpp si no existe
if not os.path.exists('koboldcpp_linux') or os.path.getsize('koboldcpp_linux') < 1000000:
    print('📥 Descargando KoboldCpp...')
    !wget -q -c {KOBOLD_URL} -O koboldcpp_linux
    !chmod +x koboldcpp_linux

# 5. Descargar o enlazar modelo en Google Drive
if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 5000000000:
    print('⚡ Modelo encontrado en Google Drive! Enlazando...')
    !ln -sf "{MODEL_PATH}" model.gguf
else:
    print('📥 Descargando NeuralDaredevil 8B Abliterated a Google Drive (5.7 GB - 0% Censura)...')
    !wget -c "{MODEL_URL}" -O "{MODEL_PATH}"
    !ln -sf "{MODEL_PATH}" model.gguf

# 6. Iniciar KoboldCpp en el puerto 5001 con GPU T4
print(f'🚀 Servidor listo en: https://{NGROK_DOMAIN}/v1')
!./koboldcpp_linux --model model.gguf --usecuda 0 mmq --gpulayers 999 --contextsize 8192 --port 5001 --skiplauncher
